# H&E → HER2: Variance-Preserving Conditional Flow Matching

Noise-to-HER2 generation conditioned on H&E, using the trigonometric probability path from TorchCFM's `VariancePreservingConditionalFlowMatcher`.


In [ ]:
MODE = "train"                  # "train" или "smoke"
SPLIT_POLICY = "group-disjoint"  
RUN_NAME = "her2-vp-cfm"
DATASET_OVERRIDE = None     # Например: "/kaggle/input/datasets/vasiliygorelov/mist-her2/mist-her2-flow" 
RESUME_FROM = None               # Путь к latest.pt из предыдущих Kaggle Outputs

TOTAL_STEPS = 20_000             # Общий learning-rate schedule
SESSION_STEPS = 15_000            # Сколько optimizer steps выполнить в этой сессии
CROP_SIZE = 512                  # Кроп исходного изображения, без уменьшения масштаба
CHANNELS = (32, 64, 128, 256)
BATCH_SIZE = 2
ACCUMULATION_STEPS = 4           
NUM_WORKERS = 2
SEED = 42

RUN_ABLATION = False             # Сравнение солверов только на validation
RUN_TEST = False                 # Включить после выбора модели/солвера на validation
FULL_TEST = False                # True: весь test, 1024x1024; False: небольшой preview
FEATURE_METRICS = True           
TEST_SOLVER = "heun"
TEST_STEPS = 16
PREVIEW_CROP_SIZE = 512

RUN_VISUALIZATIONS = True
VIZ_AUDIT_SAMPLES = 48            # Таргеты train/val для распределения DAB
VIZ_PAIR_INDEX = 0               # Индекс validation-патча для детального разбора
VIZ_FLOW_STEPS = 32              # Детализация траектории
VIZ_SEEDS = tuple(SEED + i for i in range(4))
DAB_THRESHOLD = 0.05             # Цветовой прокси


## Dataset and environment

Attach the prepared MIST-HER2 Kaggle bundle. The setup cell finds `kaggle_bundle.json` and imports `ihc_flow` from its `code/` directory. Outputs are written to `/kaggle/working`.


In [ ]:
import importlib.util
import json
import os
from pathlib import Path
import shutil
import stat
import subprocess
import sys
import tempfile
import zipfile

INPUT_ROOT = Path(os.environ.get("IHC_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("IHC_WORK_ROOT", "/kaggle/working"))
WORK_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(WORK_ROOT / ".matplotlib"))

def locate_bundle():
    if DATASET_OVERRIDE:
        root = Path(DATASET_OVERRIDE)
        if not (root / "kaggle_bundle.json").is_file():
            raise FileNotFoundError(f"Неверный DATASET_OVERRIDE: {root}")
        return root
    matches = list(INPUT_ROOT.rglob("kaggle_bundle.json"))
    if not matches:
        archives = list(INPUT_ROOT.rglob("mist-her2-flow-kaggle.zip"))
        if len(archives) == 1:
            target = Path(tempfile.gettempdir()) / "ihc-flow-dataset"
            target.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(archives[0]) as archive:
                for member in archive.infolist():
                    destination = (target / member.filename).resolve()
                    if not destination.is_relative_to(target.resolve()):
                        raise ValueError(f"Unsafe archive path: {member.filename}")
                    if stat.S_ISLNK(member.external_attr >> 16):
                        raise ValueError("Archive symlinks are unsupported")
                archive.extractall(target)
            matches = list(target.rglob("kaggle_bundle.json"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Найдено {len(matches)} dataset bundles. Подключите нужный dataset "
            "или укажите DATASET_OVERRIDE."
        )
    return matches[0].parent

BUNDLE = locate_bundle()
required = ("torch", "torchvision", "numpy", "PIL", "yaml", "scipy", "skimage", "tqdm", "matplotlib")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"В базовом окружении отсутствуют пакеты: {missing}. Используйте Kaggle Python image.")
if sys.version_info < (3, 11):
    raise RuntimeError("Требуется Python 3.11 или новее")

if importlib.util.find_spec("pytorch_fid") is None:
    dependencies = WORK_ROOT / "dependencies"
    wheel = next((BUNDLE / "wheels").glob("pytorch_fid-*.whl"))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                           "--no-cache-dir", "--no-index", "--no-deps",
                           "--target", str(dependencies), str(wheel)])
    sys.path.insert(0, str(dependencies))
sys.path.insert(0, str(BUNDLE / "code"))

# BEGIN IHC AMP RUNTIME PATCH
# Copy only Python code into Working. Dataset images stay in read-only Input.
runtime_code = WORK_ROOT / "ihc-runtime"
runtime_package = runtime_code / "ihc_flow"
shutil.copytree(BUNDLE / "code" / "ihc_flow", runtime_package,
                dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__"))
patched_training_source = '"""Step-based training, EMA, validation-only model selection, and resumable checkpoints."""\n\nfrom __future__ import annotations\n\nimport copy\nimport json\nimport math\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom skimage.metrics import structural_similarity\nfrom torch import nn\nfrom torch.utils.data import DataLoader\nfrom tqdm.auto import tqdm\n\nfrom .config import Config\nfrom .data import PairedDataset, TrainingBatches\nfrom .download import sha256_file\nfrom .flow import integrate\nfrom .model import ConditionalUNet\nfrom .runtime import environment_info, initial_noise, save_image, seed_everything, write_json\n\nFLOW_VARIANT = \"variance_preserving_trigonometric\"\n\ndef variance_preserving_flow_matching_loss(model, condition, target, noise, time):\n    \"\"\"TorchCFM-compatible VP-CFM objective for x0=noise and x1=target.\"\"\"\n    time_view = time.reshape(-1, *([1] * (target.dim() - 1)))\n    angle = (math.pi / 2) * time_view\n    cosine, sine = torch.cos(angle), torch.sin(angle)\n    state = cosine * noise + sine * target\n    target_velocity = (math.pi / 2) * (cosine * target - sine * noise)\n    predicted_velocity = model(state, time, condition)\n    return torch.nn.functional.mse_loss(predicted_velocity, target_velocity)\n\n\n@torch.no_grad()\ndef update_ema(ema: nn.Module, model: nn.Module, decay: float) -> None:\n    for averaged, current in zip(ema.parameters(), model.parameters(), strict=True):\n        averaged.lerp_(current, 1 - decay)\n    for averaged, current in zip(ema.buffers(), model.buffers(), strict=True):\n        averaged.copy_(current)\n\n\n@torch.no_grad()\ndef optimizer_update(\n    model: nn.Module,\n    optimizer: torch.optim.Optimizer,\n    scaler: torch.amp.GradScaler,\n    max_norm: float,\n) -> tuple[bool, float | None]:\n    """Apply one effective-batch update, or let AMP safely skip overflowing gradients.\n\n    Gradient clipping must not prevent GradScaler from lowering its loss scale.\n    A skipped update must also skip EMA, the optimizer-step counter and LR schedule.\n    """\n    scaler.unscale_(optimizer)\n    parameters = [parameter for parameter in model.parameters() if parameter.grad is not None]\n    if not parameters:\n        raise RuntimeError("No gradients were produced for the optimizer update")\n    try:\n        norm = nn.utils.clip_grad_norm_(parameters, max_norm, error_if_nonfinite=True)\n    except RuntimeError as error:\n        if "non-finite" not in str(error):\n            raise  # Do not swallow unrelated failures, including device/memory errors.\n        finite = torch.stack([torch.isfinite(p.grad).all() for p in parameters]).all().item()\n        if finite:\n            # Finite FP32 gradients can still overflow a norm reduction. Clip in FP64.\n            norm_device = parameters[0].device if parameters[0].device.type != "mps" else "cpu"\n            norm = torch.linalg.vector_norm(\n                torch.stack(\n                    [\n                        torch.linalg.vector_norm(p.grad.to(device=norm_device, dtype=torch.float64))\n                        for p in parameters\n                    ]\n                )\n            )\n            if not torch.isfinite(norm):\n                raise FloatingPointError("Gradient norm remains non-finite in FP64") from error\n            coefficient = (max_norm / (norm + 1e-6)).clamp(max=1)\n            for parameter in parameters:\n                clipped = parameter.grad.to(device=norm_device, dtype=torch.float64) * coefficient\n                parameter.grad.copy_(clipped.to(parameter.grad))\n        elif not scaler.is_enabled():\n            raise FloatingPointError("Non-finite gradients with AMP disabled") from error\n        else:\n            # unscale_ already recorded inf/NaN; step() will not touch the parameters.\n            scaler.step(optimizer)\n            scaler.update()\n            optimizer.zero_grad(set_to_none=True)\n            return False, None\n    scaler.step(optimizer)\n    scaler.update()\n    return True, float(norm.item())\n\n\ndef learning_rate(step: int, config: Config) -> float:\n    settings = config.train\n    if step < settings.warmup_steps:\n        factor = (step + 1) / max(1, settings.warmup_steps)\n    else:\n        progress = (step - settings.warmup_steps) / max(\n            1, settings.max_steps - settings.warmup_steps\n        )\n        factor = 0.5 * (1 + math.cos(math.pi * min(progress, 1)))\n    return settings.learning_rate * factor\n\n\ndef save_checkpoint(path: Path, payload: dict) -> None:\n    temporary = path.with_suffix(".tmp")\n    torch.save(payload, temporary)\n    os.replace(temporary, path)\n\n\n@torch.inference_mode()\ndef validate(\n    model: nn.Module, loader: DataLoader, device: torch.device, config: Config, output: Path\n) -> dict:\n    model.eval()\n    settings = config.train\n    errors, similarities = [], []\n    for batch_index, batch in enumerate(loader):\n        condition, target = batch["he"].to(device), batch["ihc"].to(device)\n        noise = initial_noise(condition, batch["id"], settings.seed)\n        prediction = integrate(\n            model, noise, condition, settings.validation_steps, settings.validation_solver\n        ).clamp(-1, 1)\n        errors.extend((prediction - target).square().flatten(1).mean(1).cpu().tolist())\n        for predicted, real in zip(prediction, target, strict=True):\n            first = predicted.cpu().permute(1, 2, 0).numpy()\n            second = real.cpu().permute(1, 2, 0).numpy()\n            similarities.append(structural_similarity(first, second, data_range=2, channel_axis=-1))\n        if batch_index == 0:\n            for index in range(min(4, len(condition))):\n                panel = torch.cat((condition[index], prediction[index], target[index]), dim=2)\n                save_image(panel, output / f"example-{index}.png")\n    return {\n        "val_mse": float(np.mean(errors)),\n        "val_ssim": float(np.mean(similarities)),\n        "count": len(errors),\n        "solver": settings.validation_solver,\n        "steps": settings.validation_steps,\n        "crop_size": settings.crop_size,\n    }\n\n\ndef train(\n    config: Config,\n    root: Path,\n    manifest: Path,\n    output: Path,\n    device: torch.device,\n    resume: Path | None = None,\n    stop_after: int | None = None,\n    limit_train: int | None = None,\n    progress: bool = False,\n) -> Path:\n    settings = config.train\n    if output.exists() and any(output.iterdir()) and resume is None:\n        raise FileExistsError(\n            f"Run directory is not empty: {output}; use --resume or a new directory"\n        )\n    output.mkdir(parents=True, exist_ok=True)\n    seed_everything(settings.seed)\n    model = ConditionalUNet(config.model).to(device)\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    optimizer = torch.optim.AdamW(\n        model.parameters(), lr=settings.learning_rate, weight_decay=settings.weight_decay\n    )\n    use_amp = settings.mixed_precision and device.type == "cuda"\n    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)\n    noise_generator = torch.Generator().manual_seed(settings.seed + 1)\n    step, batches_seen, best_ssim = 0, 0, -1.0\n    amp_skipped_updates, consecutive_amp_skips = 0, 0\n    fingerprint = sha256_file(manifest)\n    if resume is not None:\n        state = torch.load(resume, map_location="cpu", weights_only=True)\n        if state["config"] != config.to_dict() or state["manifest_sha256"] != fingerprint:\n            raise ValueError("Resume requires the same config and manifest as the checkpoint")\n        if state["limit_train"] != limit_train:\n            raise ValueError("Resume requires the same --limit-train")\n        if state.get("flow_variant") != FLOW_VARIANT:\n            raise ValueError("Checkpoint was created with a different Flow Matching path")\n        model.load_state_dict(state["model"])\n        ema.load_state_dict(state["ema"])\n        optimizer.load_state_dict(state["optimizer"])\n        scaler.load_state_dict(state["scaler"])\n        step, batches_seen, best_ssim = state["step"], state["batches_seen"], state["best_ssim"]\n        amp_skipped_updates = state.get("amp_skipped_updates", 0)\n        noise_generator.set_state(state["noise_rng"])\n        torch.set_rng_state(state["torch_rng"])\n        if device.type == "cuda" and state["cuda_rng"] is not None:\n            # Kaggle may assign one GPU in the next session after a two-GPU session.\n            torch.cuda.set_rng_state_all(state["cuda_rng"][: torch.cuda.device_count()])\n    dataset = PairedDataset(root, manifest, "train", settings.crop_size, True, limit_train)\n    validation = PairedDataset(root, manifest, "val", settings.crop_size)\n    if {r["group"] for r in dataset.records} & {r["group"] for r in validation.records}:\n        raise ValueError("Training and validation groups overlap; rebuild the manifest")\n    # Spread the fixed validation subset across the whole holdout instead of taking its first slide.\n    indices = np.linspace(\n        0, len(validation) - 1, min(settings.validation_samples, len(validation)), dtype=int\n    )\n    validation.records = [validation.records[index] for index in indices]\n    loader = DataLoader(\n        dataset,\n        batch_sampler=TrainingBatches(\n            len(dataset), settings.batch_size, settings.seed, batches_seen\n        ),\n        num_workers=settings.num_workers,\n        pin_memory=device.type == "cuda",\n        persistent_workers=settings.num_workers > 0,\n        generator=torch.Generator().manual_seed(settings.seed + 2),\n    )\n    val_loader = DataLoader(\n        validation,\n        batch_size=settings.batch_size,\n        num_workers=0,\n        generator=torch.Generator().manual_seed(settings.seed + 3),\n    )\n    iterator = iter(loader)\n    write_json(output / "config.json", config.to_dict())\n    write_json(\n        output / "run.json",\n        {\n            **environment_info(device),\n            "manifest": str(manifest.resolve()),\n            "manifest_sha256": fingerprint,\n            "train_pairs": len(dataset),\n            "validation_pairs": len(validation),\n            "limit_train": limit_train,\n            "parameters": sum(p.numel() for p in model.parameters()),\n            "selection": "Maximum SSIM on a fixed subset of the validation split, EMA weights",\n            "amp_nonfinite_policy": "Skip overflowing updates; EMA/LR/step advance only on success",\n            "flow_variant": FLOW_VARIANT,\n        },\n    )\n    final_step = (\n        min(settings.max_steps, stop_after) if stop_after is not None else settings.max_steps\n    )\n    if final_step <= step:\n        raise ValueError("Requested stopping step must be greater than the resumed step")\n    latest = output / "latest.pt"\n\n    def payload() -> dict:\n        return {\n            "format_version": 1,\n            "config": config.to_dict(),\n            "model": model.state_dict(),\n            "ema": ema.state_dict(),\n            "optimizer": optimizer.state_dict(),\n            "scaler": scaler.state_dict(),\n            "step": step,\n            "batches_seen": batches_seen,\n            "best_ssim": best_ssim,\n            "noise_rng": noise_generator.get_state(),\n            "torch_rng": torch.get_rng_state(),\n            "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,\n            "manifest_sha256": fingerprint,\n            "limit_train": limit_train,\n            "amp_skipped_updates": amp_skipped_updates,\n            "flow_variant": FLOW_VARIANT,\n        }\n\n    bar = tqdm(total=final_step, initial=step, desc="Optimizer steps", disable=not progress)\n    while step < final_step:\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        lr = learning_rate(step, config)\n        for group in optimizer.param_groups:\n            group["lr"] = lr\n        loss_sum = 0.0\n        for _ in range(settings.accumulation_steps):\n            batch = next(iterator)\n            batches_seen += 1\n            condition, target = batch["he"].to(device), batch["ihc"].to(device)\n            noise = torch.randn(target.shape, generator=noise_generator).to(device)\n            time = torch.rand(len(target), generator=noise_generator).to(device)\n            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):\n                loss = variance_preserving_flow_matching_loss(model, condition, target, noise, time)\n            if not torch.isfinite(loss):\n                save_checkpoint(output / "last-safe.pt", payload())\n                raise FloatingPointError(f"Non-finite training loss at step {step + 1}")\n            scaler.scale(loss / settings.accumulation_steps).backward()\n            loss_sum += loss.detach().item() / settings.accumulation_steps\n        scale_before = scaler.get_scale()\n        try:\n            updated, grad_norm = optimizer_update(model, optimizer, scaler, settings.grad_clip)\n        except FloatingPointError:\n            save_checkpoint(output / "last-safe.pt", payload())\n            raise\n        if not updated:\n            amp_skipped_updates += 1\n            consecutive_amp_skips += 1\n            event = {\n                "event": "amp_overflow_skipped",\n                "step": step,\n                "batches_seen": batches_seen,\n                "scale_before": scale_before,\n                "scale_after": scaler.get_scale(),\n                "consecutive_skips": consecutive_amp_skips,\n            }\n            with (output / "amp_events.jsonl").open("a") as stream:\n                stream.write(json.dumps(event) + "\\n")\n            print(json.dumps(event), flush=True)\n            if consecutive_amp_skips >= 16:\n                save_checkpoint(output / "last-safe.pt", payload())\n                raise FloatingPointError(\n                    "16 consecutive AMP overflows despite scale reduction; last-safe.pt saved. "\n                    "Inspect numerical stability rather than continuing to skip indefinitely."\n                )\n            continue\n        consecutive_amp_skips = 0\n        update_ema(ema, model, settings.ema_decay)\n        step += 1\n        bar.update(1)\n        bar.set_postfix(loss=f"{loss_sum:.4f}", lr=f"{lr:.2g}", refresh=False)\n        record = {\n            "step": step,\n            "fm_loss": loss_sum,\n            "learning_rate": lr,\n            "grad_norm": grad_norm,\n            "amp_scale": scaler.get_scale(),\n            "amp_skipped_updates": amp_skipped_updates,\n        }\n        if step % settings.validate_every == 0 or step == final_step:\n            metrics = validate(ema, val_loader, device, config, output / "validation" / str(step))\n            record.update(metrics)\n            if metrics["val_ssim"] > best_ssim:\n                best_ssim = metrics["val_ssim"]\n                save_checkpoint(output / "best.pt", payload())\n        if step % settings.checkpoint_every == 0 or step == final_step:\n            save_checkpoint(latest, payload())\n        with (output / "history.jsonl").open("a") as stream:\n            stream.write(json.dumps(record) + "\\n")\n        if step % settings.log_every == 0 or step == final_step:\n            print(json.dumps(record), flush=True)\n    bar.close()\n    return latest\n\n\ndef load_model(checkpoint: Path, device: torch.device) -> tuple[ConditionalUNet, Config]:\n    state = torch.load(checkpoint, map_location="cpu", weights_only=True)\n    config = Config.from_dict(state["config"])\n    model = ConditionalUNet(config.model).to(device)\n    model.load_state_dict(state["ema"])\n    return model.eval(), config\n'
(runtime_package / "train.py").write_text(patched_training_source)
sys.path.insert(0, str(runtime_code))
# Re-import the patched module even when this cell is rerun in the same kernel.
for module_name in tuple(sys.modules):
    if module_name == "ihc_flow" or module_name.startswith("ihc_flow."):
        del sys.modules[module_name]
print("Training runtime: AMP overflow recovery enabled; flow path: variance-preserving")
# END IHC AMP RUNTIME PATCH


import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image
from ihc_flow.config import Config, ModelConfig, TrainConfig
from ihc_flow.data import PairedDataset, read_records
from ihc_flow.download import sha256_file
from ihc_flow.flow import integrate
from ihc_flow.inference import generate
from ihc_flow.metrics import evaluate
from ihc_flow.model import ConditionalUNet
from ihc_flow.runtime import initial_noise, save_image, write_json
from ihc_flow.train import load_model, train
from ihc_flow.experiments import ablate
from ihc_flow.biology import analyze_biology

if MODE not in ("train", "smoke") or SPLIT_POLICY not in ("official", "group-disjoint"):
    raise ValueError("Неверный MODE или SPLIT_POLICY")
if MODE == "train" and not torch.cuda.is_available():
    raise RuntimeError("Включите GPU в Kaggle Settings или выберите MODE='smoke'")
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(4)

CACHE_DIR = Path(tempfile.gettempdir()) / "ihc-flow-metric-cache"
weights_dir = CACHE_DIR / "torch" / "checkpoints"
weights_dir.mkdir(parents=True, exist_ok=True)
for source in (BUNDLE / "metric_weights" / "torch" / "checkpoints").glob("*.pth"):
    target = weights_dir / source.name
    if not target.exists():
        target.symlink_to(source.resolve())

DATA_ROOT = BUNDLE / "data" / "HER2"
MANIFEST = BUNDLE / "manifests" / SPLIT_POLICY / "pairs.jsonl"
suffix = f"{SPLIT_POLICY}-{'smoke' if MODE == 'smoke' else 'train'}"
RUN_DIR = WORK_ROOT / "runs" / f"{RUN_NAME}-{suffix}"
print("Flow variant: variance-preserving trigonometric path")
print("Dataset:", BUNDLE)
print("Output:", RUN_DIR)
print("PyTorch:", torch.__version__, "Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(DEVICE))
    print("VRAM:", round(torch.cuda.get_device_properties(DEVICE).total_memory / 2**30, 1), "GiB")

### Figure settings

Control optional figures with the visualization flags in the configuration cell.


In [ ]:
"""Source embedded directly into the notebook, compatible with the existing dataset ZIP."""

import csv
from collections import Counter

from matplotlib.patches import FancyBboxPatch
from skimage.color import rgb2hed

FIGURE_DIR = WORK_ROOT / "figures" / RUN_DIR.name
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def show_figure(figure, name):
    figure.savefig(FIGURE_DIR / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(figure)


def rgb_array(tensor):
    return (tensor.detach().float().cpu().clamp(-1, 1).permute(1, 2, 0).numpy() + 1) / 2


def dab_data(image):
    rgb = image / 255.0 if image.dtype == np.uint8 else image
    tissue = np.min(rgb, axis=-1) < 235 / 255
    concentration = rgb2hed(rgb)[..., 2]
    fraction = float(((concentration > DAB_THRESHOLD) & tissue).sum() / max(tissue.sum(), 1))
    return concentration, tissue, fraction


def read_csv_rows(path):
    with path.open(newline="") as stream:
        return list(csv.DictReader(stream))


def audit_figures():
    records = {split: read_records(MANIFEST, split) for split in ("train", "val", "test")}
    groups = {split: {record["group"] for record in rows} for split, rows in records.items()}
    labels = list(records)
    overlap = np.array([[len(groups[a] & groups[b]) for b in labels] for a in labels])
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].bar(labels, [len(records[label]) for label in labels], color=["#3974ac", "#df9850", "#66a080"])
    axes[0].set(title="Pairs in each split", ylabel="Number of pairs")
    for index, label in enumerate(labels):
        axes[0].text(index, len(records[label]), str(len(records[label])), ha="center", va="bottom")
    heat = axes[1].imshow(overlap, cmap="Blues")
    axes[1].set(xticks=range(3), yticks=range(3), xticklabels=labels, yticklabels=labels,
                title="Shared filename groups")
    for row in range(3):
        for col in range(3):
            axes[1].text(col, row, str(overlap[row, col]), ha="center", va="center",
                         color="white" if overlap[row, col] > overlap.max() / 2 else "black")
    fig.colorbar(heat, ax=axes[1], fraction=0.05)
    for label in labels:
        counts = list(Counter(record["group"] for record in records[label]).values())
        axes[2].hist(counts, bins=12, alpha=0.45, label=label)
    axes[2].set(title="Pairs per filename group", xlabel="Pairs", ylabel="Groups")
    axes[2].legend()
    fig.tight_layout()
    show_figure(fig, "dataset-splits-and-groups")

    # Target staining statistics use train/validation only; test remains untouched here.
    rng = np.random.default_rng(SEED)
    distributions = {}
    for split in ("train", "val"):
        selected = rng.choice(len(records[split]), min(VIZ_AUDIT_SAMPLES, len(records[split])), replace=False)
        fractions = []
        for index in selected:
            with Image.open(DATA_ROOT / records[split][int(index)]["ihc"]) as image:
                fractions.append(dab_data(np.array(image.convert("RGB")))[2])
        distributions[split] = fractions
    fig, ax = plt.subplots(figsize=(8, 3.8))
    for split, values in distributions.items():
        ax.hist(values, bins=np.linspace(0, 1, 21), alpha=0.5, label=f"{split}: n={len(values)}")
    ax.set(xlabel="DAB-positive tissue fraction", ylabel="Sampled patches",
           title=f"Target staining distribution; fixed DAB threshold={DAB_THRESHOLD}")
    ax.legend()
    fig.tight_layout()
    show_figure(fig, "target-staining-distribution")


def method_diagram():
    fig, axes = plt.subplots(2, 1, figsize=(14, 5))
    labels = [
        ["Real HER2 y + noise z", "VP state x_t", "U-Net(x_t, t, H&E)", "Velocity MSE vs VP target"],
        ["Noise x_0", "U-Net(x_t, t, H&E)", "ODE step x_t → x_(t+dt)", "Virtual HER2 at t=1"],
    ]
    colors = ["#edf2f7", "#e1effa", "#e8f2e8", "#f8e9d8"]
    for row, ax in enumerate(axes):
        ax.set(xlim=(0, 1), ylim=(0, 1))
        ax.axis("off")
        ax.text(0.02, 0.93, "TRAINING: no ODE integration" if row == 0 else
                "GENERATION: H&E stays fixed; real IHC is unavailable", fontsize=11, weight="bold")
        for column, label in enumerate(labels[row]):
            x = 0.02 + column * 0.25
            box = FancyBboxPatch((x, 0.38), 0.20, 0.30, boxstyle="round,pad=0.008",
                                 facecolor=colors[column], edgecolor="#526579")
            ax.add_patch(box)
            ax.text(x + 0.1, 0.53, label, ha="center", va="center", fontsize=9.5)
            if column < 3:
                ax.annotate("", xy=(x + 0.244, 0.53), xytext=(x + 0.205, 0.53),
                            arrowprops={"arrowstyle": "->", "color": "#526579"})
        if row == 1:
            ax.annotate("", xy=(0.37, 0.36), xytext=(0.62, 0.36),
                        arrowprops={"arrowstyle": "->", "connectionstyle": "arc3,rad=-0.4"})
            ax.text(0.5, 0.12, "Repeat until t=1", ha="center", fontsize=10)
    fig.tight_layout()
    show_figure(fig, "method-overview")


def learning_figures(history):
    if not history:
        print("История обучения пока пуста.")
        return
    steps = np.array([row["step"] for row in history])
    losses = np.array([row["fm_loss"] for row in history])
    window = max(1, min(30, len(history) // 10))
    smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")
    validation = [row for row in history if "val_ssim" in row]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(steps, losses, alpha=0.3, color="#3974ac", label="Raw loss")
    axes[0].plot(steps[window - 1:], smoothed, color="#3974ac", label=f"Mean over {window} logs")
    axes[0].set(xlabel="Optimizer step", ylabel="Velocity MSE", title="Flow matching objective")
    axes[0].legend()
    axes[1].plot([r["step"] for r in validation], [r["val_ssim"] for r in validation], "o-", color="#66a080")
    if validation:
        best = max(validation, key=lambda row: row["val_ssim"])
        axes[1].scatter(best["step"], best["val_ssim"], marker="*", s=170, color="#df9850", zorder=4)
    axes[1].set(xlabel="Optimizer step", ylabel="SSIM", title="Fixed validation subset")
    axes[2].plot(steps, [row["learning_rate"] for row in history], color="#ab6c99")
    axes[2].set(xlabel="Optimizer step", ylabel="Learning rate", title="Warmup / cosine schedule")
    if start_step:
        for ax in axes:
            ax.axvline(start_step, color="gray", ls="--", lw=1, label="Resumed session")
    for ax in axes:
        ax.grid(alpha=0.2)
    fig.tight_layout()
    show_figure(fig, "learning-curves")


def prediction_diagnostics(pair, predicted):
    source, real, fake = rgb_array(pair["he"]), rgb_array(pair["ihc"]), rgb_array(predicted)
    error = np.abs(fake - real).mean(axis=-1)
    real_dab, _, real_fraction = dab_data(real)
    fake_dab, _, fake_fraction = dab_data(fake)
    maximum = max(float(np.quantile(np.concatenate((real_dab.ravel(), fake_dab.ravel())), 0.99)), 0.01)
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    for ax, image, label in zip(axes[0], (source, real, fake), ("H&E condition", "Real HER2", "Predicted HER2"), strict=True):
        ax.imshow(image)
        ax.set_title(label)
        ax.axis("off")
    image = axes[1, 0].imshow(error, cmap="inferno", vmin=0, vmax=1)
    axes[1, 0].set_title("Absolute RGB error (0–1)")
    fig.colorbar(image, ax=axes[1, 0], fraction=0.04)
    for ax, values, fraction, label in ((axes[1, 1], real_dab, real_fraction, "Real DAB"),
                                         (axes[1, 2], fake_dab, fake_fraction, "Predicted DAB")):
        image = ax.imshow(values, cmap="YlOrBr", vmin=0, vmax=maximum)
        ax.set_title(f"{label}; positive tissue={fraction:.1%}")
        fig.colorbar(image, ax=ax, fraction=0.04, label="Color deconvolution signal")
    for ax in axes[1]:
        ax.axis("off")
    fig.suptitle(pair["id"] + " | same DAB color scale; 99th-percentile display limit", fontsize=11)
    fig.tight_layout()
    show_figure(fig, "validation-error-and-dab")


class TimeWindowVelocity(torch.nn.Module):
    """Reuse the bundled solver on [0,1] to integrate a physical time window [a,a+dt]."""

    def __init__(self, model, start, delta):
        super().__init__()
        self.model, self.start, self.delta = model, start, delta

    def forward(self, image, time, condition):
        return self.delta * self.model(image, self.start + self.delta * time, condition)


@torch.inference_mode()
def trace_flow(model, noise, condition, steps, solver):
    if steps < 1:
        raise ValueError("VIZ_FLOW_STEPS must be positive")
    indices = sorted(set(np.rint(np.linspace(0, steps, min(6, steps + 1))).astype(int)))
    states = {0: noise[0].cpu().clone()}
    state = noise.clone()
    speeds, outside = [], [float((state.abs() > 1).float().mean().item())]
    delta = 1 / steps
    for index in range(steps):
        previous = state
        window = TimeWindowVelocity(model, index * delta, delta)
        state = integrate(window, previous, condition, steps=1, solver=solver)
        speeds.append(float((state - previous).square().mean().sqrt().item() / delta))
        outside.append(float((state.abs() > 1).float().mean().item()))
        if index + 1 in indices:
            states[index + 1] = state[0].cpu().clone()
    return indices, states, speeds, outside, state


def trajectory_figures(pair, indices, states, speeds, outside, steps, solver, noise):
    fig, axes = plt.subplots(2, len(indices), figsize=(15, 5.5), squeeze=False)
    for column, index in enumerate(indices):
        time = index / steps
        angle = np.pi * time / 2
        reference = np.cos(angle) * noise[0].cpu() + np.sin(angle) * pair["ihc"]
        for row, tensor in enumerate((reference, states[index])):
            axes[row, column].imshow(rgb_array(tensor))
            axes[row, column].set_title(f"t={time:.2f}")
            axes[row, column].set(xticks=[], yticks=[])
    axes[0, 0].set_ylabel("VP training reference\nnoise → real HER2", fontsize=10)
    axes[1, 0].set_ylabel("Learned ODE\nH&E fixed", fontsize=10)
    mode_note = "Smoke check | " if MODE == "smoke" else ""
    fig.suptitle(f"{mode_note}{solver}: {steps} steps; RGB clipping is for display only", fontsize=12)
    fig.tight_layout(h_pad=3.0, w_pad=0.4, rect=(0, 0, 1, 0.93))
    show_figure(fig, "flow-reference-vs-sampling")
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].plot((np.arange(steps) + 0.5) / steps, speeds, "o-")
    axes[0].set(xlabel="ODE time", ylabel="RMS displacement / dt", title="Mean motion per step")
    axes[1].plot(np.arange(steps + 1) / steps, outside, "o-", color="#df9850")
    axes[1].set(xlabel="ODE time", ylabel="Fraction outside [-1,1]", ylim=(0, 1), title="Unclipped state values")
    fig.tight_layout()
    show_figure(fig, "flow-state-diagnostics")


def seed_figures(pair, predictions, seeds):
    samples = np.stack([rgb_array(prediction) for prediction in predictions])
    spread = samples.std(axis=0).mean(axis=-1)
    count = len(predictions)
    fig, axes = plt.subplots(1, count + 1, figsize=(3 * (count + 1), 3.5))
    for ax, image, seed in zip(axes[:-1], samples, seeds, strict=True):
        ax.imshow(image)
        ax.set_title(f"Seed {seed}")
        ax.axis("off")
    heat = axes[-1].imshow(spread, cmap="magma", vmin=0, vmax=0.5)
    axes[-1].set_title("Mean channel std")
    axes[-1].axis("off")
    fig.colorbar(heat, ax=axes[-1], fraction=0.04)
    fig.suptitle("Same H&E / checkpoint / solver; seed variability is not calibrated uncertainty")
    fig.tight_layout()
    show_figure(fig, "seed-variability")


def ablation_figures(rows):
    metrics = [("ssim", "SSIM ↑", False)]
    metrics += [(key, label, True) for key, label in (("fid", "FID ↓"), ("kid_x1000", "KID ×1000 ↓"))
                if any(row.get(key) is not None for row in rows)]
    fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 4), squeeze=False)
    for ax, (key, label, _) in zip(axes[0], metrics, strict=True):
        for solver in ("euler", "heun", "rk4"):
            selected = sorted([row for row in rows if row["solver"] == solver and row.get(key) is not None], key=lambda r: r["nfe"])
            ax.plot([r["seconds_per_image"] for r in selected], [r[key] for r in selected], "o-", label=solver)
            for row in selected:
                ax.annotate(str(row["nfe"]), (row["seconds_per_image"], row[key]), xytext=(4, 4), textcoords="offset points", fontsize=8)
        ax.set(xlabel="Seconds / image", ylabel=label, title=label + " vs inference time")
        ax.grid(alpha=0.2)
        ax.legend()
    fig.suptitle("Validation only; point labels = NFE; cropped subset ≠ full benchmark", fontsize=11)
    fig.tight_layout()
    show_figure(fig, "ode-time-quality")
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for solver in ("euler", "heun", "rk4"):
        selected = sorted([r for r in rows if r["solver"] == solver], key=lambda r: r["nfe"])
        for ax, key in ((axes[0], "ssim"), (axes[1], "seconds_per_image")):
            ax.plot([r["nfe"] for r in selected], [r[key] for r in selected], "o-", label=solver)
    axes[0].set(xlabel="Network evaluations (NFE)", ylabel="SSIM", title="Quality at equal NFE")
    axes[1].set(xlabel="Network evaluations (NFE)", ylabel="Seconds / image", title="Runtime at equal NFE")
    for ax in axes:
        ax.legend()
        ax.grid(alpha=0.2)
    fig.tight_layout()
    show_figure(fig, "ode-equal-nfe")


def test_figures(test_dir):
    metrics = read_csv_rows(test_dir / "evaluation" / "per_image.csv")
    biology = read_csv_rows(test_dir / "biology" / "biology.csv")
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist([float(row["ssim"]) for row in metrics], bins=min(20, len(metrics)), color="#3974ac")
    axes[0].set(xlabel="Per-image SSIM", ylabel="Patches", title=f"Test subset: n={len(metrics)}")
    x = np.array([float(row["real_dab_positive_tissue_fraction"]) for row in biology])
    y = np.array([float(row["pred_dab_positive_tissue_fraction"]) for row in biology])
    axes[1].scatter(x, y, alpha=0.55, s=18)
    axes[1].plot([0, 1], [0, 1], "--", color="gray")
    axes[1].set(xlabel="Real DAB-positive tissue fraction", ylabel="Predicted fraction",
                xlim=(0, 1), ylim=(0, 1), title="Regional stain agreement")
    axes[2].hist(np.abs(x - y), bins=min(20, len(biology)), color="#df9850")
    axes[2].set(xlabel="Absolute DAB fraction error", ylabel="Patches", title="Failures hidden by averages")
    fig.tight_layout()
    show_figure(fig, "test-metric-distributions")


## Splits and paired examples

Check the manifest counts and group overlap before training. Adjacent H&E and IHC sections may differ locally.


In [ ]:
audit = json.loads((MANIFEST.parent / "audit.json").read_text())
print("Pairs:", audit["counts"])
print("Groups:", audit["group_counts"])
print("Group overlaps:", {key: len(value) for key, value in audit["group_overlap"].items()})
if SPLIT_POLICY == "group-disjoint":
    assert all(not value for value in audit["group_overlap"].values())
else:
    print("Официальный test содержит общие группы с train; это не оценка на новых пациентах.")

pairs = PairedDataset(DATA_ROOT, MANIFEST, "train")
fig, axes = plt.subplots(3, 2, figsize=(10, 14))
for row, index in enumerate(np.linspace(0, len(pairs) - 1, 3, dtype=int)):
    item = pairs[int(index)]
    for column, key in enumerate(("he", "ihc")):
        array = (item[key].permute(1, 2, 0).numpy() + 1) / 2
        axes[row, column].imshow(array)
        axes[row, column].set_title(("H&E: input" if key == "he" else "HER2: real target")
                                     + "\n" + item["id"])
        axes[row, column].axis("off")
plt.tight_layout()
plt.show()

### Data inspection

Inspect split sizes, patches per group and train/validation DAB distributions. Filename groups are not verified patient identities.


In [ ]:
if RUN_VISUALIZATIONS:
    if VIZ_AUDIT_SAMPLES < 1 or DAB_THRESHOLD <= 0:
        raise ValueError("VIZ_AUDIT_SAMPLES и DAB_THRESHOLD должны быть положительными")
    if MODE == "smoke":
        VIZ_AUDIT_SAMPLES = min(VIZ_AUDIT_SAMPLES, 6)
    audit_figures()

## Variance-preserving path

The U-Net uses the same noise source and fixed H&E condition, but learns a trigonometric path:

$$x_t=\cos(\pi t/2)z+\sin(\pi t/2)y.$$

$$u_t=\frac{\pi}{2}[\cos(\pi t/2)y-\sin(\pi t/2)z].$$

The network regresses this velocity. Resume only with a checkpoint from the same recipe and split.


In [ ]:
if MODE == "smoke":
    config = Config(
        ModelConfig(channels=(16, 32, 64), blocks_per_level=1),
        TrainConfig(crop_size=64, batch_size=2, accumulation_steps=1, max_steps=8,
                    learning_rate=0.001, warmup_steps=0, ema_decay=0.9, num_workers=0,
                    mixed_precision=False, log_every=1, validate_every=4, checkpoint_every=4,
                    validation_samples=4, validation_steps=4, validation_solver="euler"),
    )
    limit_train = 32
else:
    config = Config(
        ModelConfig(channels=CHANNELS, blocks_per_level=2),
        TrainConfig(crop_size=CROP_SIZE, batch_size=BATCH_SIZE,
                    accumulation_steps=ACCUMULATION_STEPS, max_steps=TOTAL_STEPS,
                    learning_rate=1e-4, warmup_steps=500, num_workers=NUM_WORKERS,
                    mixed_precision=True, log_every=100, validate_every=500,
                    checkpoint_every=200, validation_samples=16,
                    validation_steps=8, validation_solver="heun", seed=SEED),
    )
    limit_train = None

resume = Path(RESUME_FROM) if RESUME_FROM else None
if resume is None and (RUN_DIR / "latest.pt").exists():
    resume = RUN_DIR / "latest.pt"
start_step = 0
if resume is not None:
    state = torch.load(resume, map_location="cpu", weights_only=True)
    if (state["limit_train"] is not None) != (MODE == "smoke"):
        raise ValueError("MODE не совпадает с checkpoint: smoke и полное обучение имеют отдельные run.")
    if state["manifest_sha256"] != sha256_file(MANIFEST):
        raise ValueError("Checkpoint создан на другом manifest/split. Выберите исходный SPLIT_POLICY.")
    config = Config.from_dict(state["config"])
    limit_train = state["limit_train"]
    start_step = state["step"]
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    if resume.parent.resolve() != RUN_DIR.resolve():
        for filename in ("best.pt", "history.jsonl"):
            source, target = resume.parent / filename, RUN_DIR / filename
            if source.exists() and not target.exists():
                shutil.copy2(source, target)
    del state
stop_step = min(config.train.max_steps, start_step + SESSION_STEPS)
parameters = sum(p.numel() for p in ConditionalUNet(config.model).parameters())
print("Parameters:", f"{parameters:,}")
print("Optimizer steps:", start_step, "→", stop_step, "/", config.train.max_steps)
print("Crop:", config.train.crop_size, "Batch:", config.train.batch_size,
      "Accumulation:", config.train.accumulation_steps)
print("Resume:", resume)
display(config.to_dict())

### Training and inference

Training uses points between the source and real IHC. Inference uses only the source and H&E condition; the real target is never passed to the ODE solver.


In [ ]:
if RUN_VISUALIZATIONS:
    method_diagram()

## Training

The run saves checkpoints and a JSONL history. Use the saved configuration when resuming; do not change the recipe mid-run.


In [ ]:
if start_step < config.train.max_steps:
    train(config, DATA_ROOT, MANIFEST, RUN_DIR, DEVICE,
          resume=resume, stop_after=stop_step, limit_train=limit_train, progress=True)
else:
    print("Общий schedule уже завершён; обучение пропущено.")

checkpoint = RUN_DIR / "best.pt"
if not checkpoint.exists():
    checkpoint = RUN_DIR / "latest.pt"
if not checkpoint.exists() and resume is not None:
    checkpoint = resume
print("Checkpoint для генерации:", checkpoint)

## Learning curves and errors

Inspect training loss, validation SSIM, predictions and DAB maps together. Lower loss or higher SSIM alone does not establish accurate HER2 staining.


In [ ]:
history_path = RUN_DIR / "history.jsonl"
history = [json.loads(line) for line in history_path.read_text().splitlines()] if history_path.exists() else []
learning_figures(history)

model, _ = load_model(checkpoint, DEVICE)
preview_size = 64 if MODE == "smoke" else PREVIEW_CROP_SIZE
preview = PairedDataset(DATA_ROOT, MANIFEST, "val", crop_size=preview_size)
if not 0 <= VIZ_PAIR_INDEX < len(preview):
    raise ValueError(f"VIZ_PAIR_INDEX должен быть от 0 до {len(preview) - 1}")
selected_indices = np.linspace(0, len(preview) - 1, 3, dtype=int)
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
cached_predictions = {}
for row, index in enumerate(selected_indices):
    item = preview[int(index)]
    condition = item["he"].unsqueeze(0).to(DEVICE)
    noise = initial_noise(condition, [item["id"]], SEED)
    predicted = integrate(model, noise, condition, steps=4 if MODE == "smoke" else TEST_STEPS,
                          solver="euler" if MODE == "smoke" else TEST_SOLVER)[0].cpu()
    cached_predictions[int(index)] = predicted
    for column, (tensor, title) in enumerate(((item["he"], "H&E"),
                                           (predicted, "Predicted HER2"),
                                           (item["ihc"], "Real HER2"))):
        axes[row, column].imshow(rgb_array(tensor))
        axes[row, column].set_title(title)
        axes[row, column].axis("off")
plt.tight_layout()
show_figure(fig, "validation-preview")

diagnostic_pair = preview[VIZ_PAIR_INDEX]
if RUN_VISUALIZATIONS:
    predicted = cached_predictions.get(VIZ_PAIR_INDEX)
    if predicted is None:
        condition = diagnostic_pair["he"].unsqueeze(0).to(DEVICE)
        noise = initial_noise(condition, [diagnostic_pair["id"]], SEED)
        predicted = integrate(model, noise, condition, steps=4 if MODE == "smoke" else TEST_STEPS,
                              solver="euler" if MODE == "smoke" else TEST_SOLVER)[0].cpu()
    prediction_diagnostics(diagnostic_pair, predicted)
del model, cached_predictions
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

### Reference path and generated trajectory

The analytic reference path uses the real target to illustrate training. The generated trajectory uses only the initial source and fixed H&E condition.


In [ ]:
if RUN_VISUALIZATIONS:
    model, _ = load_model(checkpoint, DEVICE)
    condition = diagnostic_pair["he"].unsqueeze(0).to(DEVICE)
    noise = initial_noise(condition, [diagnostic_pair["id"]], SEED)
    flow_steps = 8 if MODE == "smoke" else VIZ_FLOW_STEPS
    flow_solver = "euler" if MODE == "smoke" else TEST_SOLVER
    indices, snapshots, speeds, outside, endpoint = trace_flow(
        model, noise, condition, flow_steps, flow_solver
    )
    if MODE == "smoke":
        # The time-window wrapper must agree with the bundled solver.
        direct = integrate(model, noise, condition, steps=flow_steps, solver=flow_solver)
        torch.testing.assert_close(endpoint, direct, rtol=1e-5, atol=1e-6)
    trajectory_figures(diagnostic_pair, indices, snapshots, speeds, outside,
                       flow_steps, flow_solver, noise)
    del model, endpoint, snapshots
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

### Seed sensitivity

Hold H&E, checkpoint and solver fixed while changing the initial seed. RGB variation is a diagnostic, not calibrated biological uncertainty.


In [ ]:
if RUN_VISUALIZATIONS:
    if len(VIZ_SEEDS) < 2 or len(set(VIZ_SEEDS)) != len(VIZ_SEEDS):
        raise ValueError("VIZ_SEEDS должен содержать хотя бы два различных seed")
    model, _ = load_model(checkpoint, DEVICE)
    condition = diagnostic_pair["he"].unsqueeze(0).to(DEVICE)
    varied_predictions = []
    for seed in VIZ_SEEDS:
        noise = initial_noise(condition, [diagnostic_pair["id"]], seed)
        prediction = integrate(model, noise, condition,
                               steps=4 if MODE == "smoke" else TEST_STEPS,
                               solver="euler" if MODE == "smoke" else TEST_SOLVER)[0]
        varied_predictions.append(prediction.cpu())
    seed_figures(diagnostic_pair, varied_predictions, VIZ_SEEDS)
    del model, varied_predictions
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## Save and resume

Save Kaggle Outputs before ending the session. Attach them as Input for the next run and set `RESUME_FROM` to the matching `latest.pt`. Keep the original dataset attached.


In [ ]:
write_json(RUN_DIR / "session.json", {
    "mode": MODE, "split_policy": SPLIT_POLICY, "start_step": start_step,
    "stop_step": stop_step, "checkpoint": str(checkpoint), "device": str(DEVICE),
    "dataset_bundle_version": json.loads((BUNDLE / "kaggle_bundle.json").read_text())["version"],
    "visualization_directory": str(FIGURE_DIR),
    "training_schedule_completed": stop_step == config.train.max_steps,
    "flow_variant": "variance_preserving_trigonometric",
})
export = WORK_ROOT / f"{RUN_DIR.name}-step{stop_step}.zip"
with zipfile.ZipFile(export, "w", compression=zipfile.ZIP_STORED) as archive:
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(RUN_DIR.parent))
    for path in sorted(FIGURE_DIR.glob("*.png")):
        archive.write(path, Path("figures") / RUN_DIR.name / path.name)
print("Run directory:", RUN_DIR)
print("Checkpoint:", RUN_DIR / "latest.pt")
print("Downloadable archive:", export)
from IPython.display import FileLink
display(FileLink(str(export)))